In [ ]:
import pickle
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp

In [ ]:
df1 = pd.read_csv('topics/f1_submission.csv')
df2 = pd.read_csv('topics/f1_comment.csv')

df1['post_created_utc'] = pd.to_datetime(df1['post_created_utc'])
df2['comment_created_utc'] = pd.to_datetime(df2['comment_created_utc'])

df1['year'] = df1['post_created_utc'].dt.year
df1['month'] = df1['post_created_utc'].dt.month
df1['timestamp'] = df1.apply(lambda x: f"{x['year']}-{x['month']:02d}", axis=1)

df2['year'] = df2['comment_created_utc'].dt.year
df2['month'] = df2['comment_created_utc'].dt.month
df2['timestamp'] = df2.apply(lambda x: f"{x['year']}-{x['month']:02d}", axis=1)

print('Number of Submission:', df1.shape[0], 'Number of columns:', df1.shape[1])
print('Number of Comments', df2.shape[0], 'Number of columns:', df2.shape[1])

In [ ]:
df1_pre = df1[df1.post_created_utc < '2020-01-01'].reset_index(drop=True)
df1_post = df1[df1.post_created_utc >= '2020-01-01'].reset_index(drop=True)

df2_pre = df2[df2.comment_created_utc < '2020-01-01'].reset_index(drop=True)
df2_post = df2[df2.comment_created_utc >= '2020-01-01'].reset_index(drop=True)

df1_pre.head(2)

In [ ]:
submission_topic_count = df1.groupby(['Name','Representation']).size().reset_index(name='count').sort_values(by='count', ascending=False)
submission_topic_count_pre = df1_pre.groupby(['Name','Representation']).size().reset_index(name='count').sort_values(by='count', ascending=False)
submission_topic_count_post = df1_post.groupby(['Name','Representation']).size().reset_index(name='count').sort_values(by='count', ascending=False)
comment_topic_count = df2.groupby(['Name','Representation']).size().reset_index(name='count').sort_values(by='count', ascending=False)
comment_topic_count_pre = df2_pre.groupby(['Name','Representation']).size().reset_index(name='count').sort_values(by='count', ascending=False)
comment_topic_count_post = df2_post.groupby(['Name','Representation']).size().reset_index(name='count').sort_values(by='count', ascending=False)

submission_topic_count

In [ ]:
# Group by year and month, count the number of topics in each group
submission_topic_counts = df2_post.groupby(['timestamp','Name']).size().reset_index(name='topic_count')

sorted_topics = sorted(submission_topic_counts['Name'].unique())

# Create subplots
fig = sp.make_subplots(rows=len(sorted_topics), cols=1, subplot_titles=sorted_topics)

# Add traces for each topic
for i, topic in enumerate(sorted_topics, start=1):
    data_topic = submission_topic_counts[submission_topic_counts['Name'] == topic]
    trace = go.Bar(
        x=data_topic['timestamp'],
        y=data_topic['topic_count'],
        # mode='lines+markers',
        # mode = 'lines',
        name=f'Topic {topic}'
    )
    fig.add_trace(trace, row=i, col=1)

# Set the same y-axis limits for all subplots
fig.update_yaxes(range=[0, submission_topic_counts['topic_count'].max()])
fig.update_xaxes(range=[submission_topic_counts['timestamp'].min(), submission_topic_counts['timestamp'].max()])

# Update layout
fig.update_layout(
    height=3000,
    width=1000,
    title_text='Distribution of Topics Over Years and Months per Topic (Histogram)',
    showlegend=False,
    xaxis_tickangle=-45
)

fig.show()

In [ ]:
submission_topic_counts = df3_post.groupby(['timestamp','Name']).size().reset_index(name='topic_count')

fig = px.line(submission_topic_counts,
                   x = 'timestamp',
                   y = 'topic_count',
                   color = 'Name')
fig.update_layout(bargap=0.2)
fig.show()

In [ ]:
def get_a_reddit_by_person(author, df):
    try:
        return df[(df['post_author'] == author) | (df['comment_author'] == author)]
    except:
        try:
            return df[df['post_author'] == author]
        except:
            return df[df['comment_author'] == author]

above5_df_pre = pd.concat(
    [get_a_reddit_by_person(author, df1_pre) for author in above5_authors],
    ignore_index=False
)
above5_df_post = pd.concat(
    [get_a_reddit_by_person(author, df1_post) for author in above5_authors],
    ignore_index=False
)

In [ ]:
above5_df_pre.post_author.unique()
len(above5_df_post.post_author.unique())